# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

In [3]:
# 🛠️ TOOL 3 (BONUS): Word Counter

def word_counter(text: str) -> int:
    """Count the number of words in a piece of text."""
    try:
        return len(text.split())
    except Exception:
        return 0

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [4]:
# 🤖 AGENT FUNCTION (IMPLEMENTED)

import re
import logging

# --- Bonus: basic logging setup ---
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("single_agent")


def agent(query: str):
    """
    Single-agent router.

    Routing rules:
      - "calculate" in query  -> Calculator Tool
      - "keywords"  in query  -> Keyword Extractor Tool
      - "count words"/"word count" in query -> Word Counter Tool (bonus)
      - otherwise              -> general/direct response

    Always returns a dict: {"type": ..., "result": ...}
    """
    if not isinstance(query, str) or not query.strip():
        logger.warning("Empty or invalid query received.")
        return {"type": "error", "result": "Query must be a non-empty string."}

    query_lower = query.lower()
    logger.info(f"Received query: {query!r}")

    try:
        # ---------- Route 1: Calculator ----------
        if "calculate" in query_lower:
            logger.info("Routing decision -> Calculator Tool")

            # Pull out a math expression from the query (numbers + operators)
            match = re.search(r"[-+]?\d[\d\s+\-*/().]*\d|[-+]?\d", query)
            if not match:
                logger.error("No valid expression found for calculation.")
                return {"type": "error", "result": "No valid mathematical expression found in query."}

            expression = match.group().strip()
            calc_result = calculator(expression)

            if calc_result == "Error in calculation":
                return {"type": "error", "result": calc_result}

            return {"type": "calculation", "result": calc_result}

        # ---------- Route 2: Keyword Extractor ----------
        elif "keywords" in query_lower:
            logger.info("Routing decision -> Keyword Extractor Tool")

            # Strip a leading instruction phrase like "extract keywords from "
            text = re.sub(r".*keywords( from)?\s*:?\s*", "", query, flags=re.IGNORECASE).strip()
            text = text if text else query

            keywords = extract_keywords(text)
            return {"type": "keywords", "result": keywords}

        # ---------- Route 3 (Bonus tool): Word Counter ----------
        elif "count words" in query_lower or "word count" in query_lower:
            logger.info("Routing decision -> Word Counter Tool")

            text = re.sub(r".*(count words|word count)( in| of| for)?\s*:?\s*", "", query, flags=re.IGNORECASE).strip()
            text = text if text else query

            count = word_counter(text)
            return {"type": "word_count", "result": count}

        # ---------- Route 4: General response ----------
        else:
            logger.info("Routing decision -> General response")
            return {
                "type": "general",
                "result": f"I don't have a specific tool for this, but here's your query noted: '{query}'. "
                           f"Try asking me to 'calculate ...' or 'extract keywords from ...'."
            }

    except Exception as e:
        logger.exception("Unhandled error while processing query.")
        return {"type": "error", "result": f"Something went wrong: {e}"}

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

### 🎁 Bonus features implemented
- **Improved routing**: uses regex to pull the actual expression/text out of the query instead of passing the whole sentence to the tool.
- **Logging**: every routing decision and error is logged via Python's `logging` module.
- **Extra tool**: added a Word Counter tool, triggered by `"count words"` or `"word count"` in the query.

In [5]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Count words in the quick brown fox jumps over the lazy dog",
    "Calculate 10 / 0",
    ""
]

for q in queries:
    print("Query:", repr(q))
    print("Response:", agent(q))
    print("-" * 50)

Query: 'Calculate 20 + 5'
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: 'Extract keywords from Artificial Intelligence is transforming industries'
Response: {'type': 'keywords', 'result': ['intelligence', 'artificial', 'transforming', 'industries']}
--------------------------------------------------
Query: 'What is machine learning?'
Response: {'type': 'general', 'result': "I don't have a specific tool for this, but here's your query noted: 'What is machine learning?'. Try asking me to 'calculate ...' or 'extract keywords from ...'."}
--------------------------------------------------
Query: 'Count words in the quick brown fox jumps over the lazy dog'
Response: {'type': 'word_count', 'result': 9}
--------------------------------------------------
Query: 'Calculate 10 / 0'
Response: {'type': 'error', 'result': 'Error in calculation'}
--------------------------------------------------
Query: ''
Response: {'type': 'error', 'res

In [6]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop): Calculate 15 * 3
Response: {'type': 'calculation', 'result': '45'}
Enter query (type 'exit' to stop): exit
